In [1]:
from datetime import datetime, timedelta
import json
import uuid
import random 
from sqlalchemy import create_engine

from utils import reset_db, get_session, model_to_dict
from data.models import cultpass

# Udahub Accounts

## Cultpass Database

**Init DB**

In [2]:
cultpass_db = "data/external/cultpass.db"

In [3]:
reset_db(cultpass_db)

✅ Removed existing data/external/cultpass.db
2026-05-29 20:10:33,360 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-05-29 20:10:33,361 INFO sqlalchemy.engine.Engine COMMIT
✅ Recreated data/external/cultpass.db with fresh schema


In [4]:
engine = create_engine(f"sqlite:///{cultpass_db}", echo=False)
cultpass.Base.metadata.create_all(engine)

**Experiences**

In [5]:
experience_data = []

with open('data/external/cultpass_experiences.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        experience_data.append(json.loads(line))

In [6]:
experience_data

[{'title': 'Carnival History Tour in Olinda',
  'description': "Discover the origins and vibrant traditions of Pernambuco's Carnival.",
  'location': 'Pernambuco, Brazil'},
 {'title': 'Sunset Paddleboarding',
  'description': 'Glide across calm waters at golden hour with all gear included.',
  'location': 'Santa Catarina, Brazil'},
 {'title': 'Pelourinho Colonial Walk',
  'description': 'Wander through colorful streets and learn about Afro-Brazilian history.',
  'location': 'Bahia, Brazil'},
 {'title': 'Samba Night at Lapa',
  'description': 'Dance the night away at a traditional samba club in the Lapa arches.',
  'location': 'Rio de Janeiro, Brazil'},
 {'title': 'Christ the Redeemer Experience',
  'description': 'Take a guided trip to one of the New Seven Wonders of the World with historical context.',
  'location': 'Rio de Janeiro, Brazil'},
 {'title': 'Modern Art at MASP',
  'description': 'Enjoy a guided visit to the São Paulo Museum of Art with insights into its top collections.',

In [7]:
with get_session(engine) as session:
    experiences = []

    for idx, experience in enumerate(experience_data):
        exp = cultpass.Experience(
            experience_id=str(uuid.uuid4())[:6],
            title=experience["title"],
            description=experience["description"],
            location=experience["location"],
            when=datetime.now() + timedelta(days=idx+1),
            slots_available=random.randint(1,30),
            is_premium=(idx % 2 == 0)
        )
        experiences.append(exp)

    session.add_all(experiences)

**User**

In [8]:
cultpass_users = []

with open('data/external/cultpass_users.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_users.append(json.loads(line))

In [9]:
cultpass_users

[{'id': 'a4ab87',
  'name': 'Alice Kingsley',
  'email': 'alice.kingsley@wonderland.com',
  'is_blocked': True},
 {'id': 'f556c0',
  'name': 'Bob Stone',
  'email': 'bob.stone@granite.com',
  'is_blocked': False},
 {'id': '88382b',
  'name': 'Cathy Bloom',
  'email': 'cathy.bloom@florals.org',
  'is_blocked': False},
 {'id': '888fb2',
  'name': 'David Noir',
  'email': 'david.noir@shadowmail.com',
  'is_blocked': True},
 {'id': 'f1f10d',
  'name': 'Eva Green',
  'email': 'eva.green@ecosoul.net',
  'is_blocked': False},
 {'id': 'e6376d',
  'name': 'Frank Ocean',
  'email': 'frank.ocean@seawaves.io',
  'is_blocked': False},
 {'id': 'b7c921',
  'name': 'Grace Hollow',
  'email': 'grace.hollow@mistmail.com',
  'is_blocked': False},
 {'id': 'c3d442',
  'name': 'Henry Vale',
  'email': 'henry.vale@ridgeview.net',
  'is_blocked': False},
 {'id': 'd8e113',
  'name': 'Isla Frost',
  'email': 'isla.frost@winterpath.org',
  'is_blocked': True},
 {'id': 'e2f774',
  'name': 'Jack Mercer',
  'email'

In [10]:
with get_session(engine) as session:
    db_users = []
    for user_info in cultpass_users:
        user = cultpass.User(
            user_id=user_info["id"],
            full_name=user_info["name"],
            email=user_info["email"],
            is_blocked=user_info["is_blocked"],
            created_at=datetime.now()
        )
        db_users.append(user)
    session.add_all(db_users) 

**Subscription**

In [11]:
with get_session(engine) as session:
    subscriptions = []
    for user_info in cultpass_users:
        subscription = cultpass.Subscription(
            subscription_id=str(uuid.uuid4())[:6],
            user_id=user_info["id"],
            status=random.choice(["active", "cancelled"]),
            tier=random.choice(["basic", "premium"]),
            monthly_quota=random.randint(2,10),
            started_at=datetime.now()
        )
        subscriptions.append(subscription)

    session.add_all(subscriptions)

**Reservation**

In [12]:
# Applicable to `cultpass_users[0]` at the moment

with get_session(engine) as session:
    experience_ids = [
        exp.experience_id
        for exp in session.query(cultpass.Experience).all()
    ]

    reservations = []
    for _ in range(40):
        reservation = cultpass.Reservation(
            reservation_id=str(uuid.uuid4()),
            user_id=random.choice(cultpass_users)["id"],
            experience_id=random.choice(experience_ids),
            status="reserved",
        )
        reservations.append(reservation)

    session.add_all(reservations)
    

In [13]:
# TODO: Add more data
# Please notice that the reservations were set to first user only 
# If you want to simulate more users later, please create more reservations per user

with get_session(engine) as session:
    reservations = session.query(cultpass.Reservation).all()

    for r in reservations:
        print({
            "reservation_id": r.reservation_id,
            "user_id": r.user_id,
            "experience_id": r.experience_id,
            "status": r.status,
        })

{'reservation_id': '937c47cc-c8b2-4e69-a48b-7f7b0fbcc064', 'user_id': 'a1d556', 'experience_id': 'da8bf5', 'status': 'reserved'}
{'reservation_id': '6a25125c-b96f-4269-9751-e1ba9aa17408', 'user_id': 'a1d490', 'experience_id': '67ca93', 'status': 'reserved'}
{'reservation_id': '8735f106-dde0-45df-990d-8360852f2b08', 'user_id': 'f556c0', 'experience_id': 'cc03b0', 'status': 'reserved'}
{'reservation_id': '235bf750-1441-4c81-b555-2b4dffb917a5', 'user_id': 'b2c989', 'experience_id': '03ab2d', 'status': 'reserved'}
{'reservation_id': 'fc1afdea-26e9-4a29-bc51-01ef743a80ab', 'user_id': 'a7da56', 'experience_id': '090d7a', 'status': 'reserved'}
{'reservation_id': '039e7525-c963-412c-90b4-13da89bbff15', 'user_id': 'eb1e90', 'experience_id': '9273e8', 'status': 'reserved'}
{'reservation_id': '1bf29a9b-82c5-4aaf-8bc7-90fca83ab892', 'user_id': 'b9e323', 'experience_id': '99ee03', 'status': 'reserved'}
{'reservation_id': '9b2d7e50-8b55-45e1-b024-e9ad33fe0f8a', 'user_id': 'f0c389', 'experience_id': 

# Tests

In [14]:
with get_session(engine) as session:
    users = session.query(cultpass.User).all()
    for user in users:
        print(user)

<User(user_id='a4ab87', email='alice.kingsley@wonderland.com', is_blocked=True)>
<User(user_id='f556c0', email='bob.stone@granite.com', is_blocked=False)>
<User(user_id='88382b', email='cathy.bloom@florals.org', is_blocked=False)>
<User(user_id='888fb2', email='david.noir@shadowmail.com', is_blocked=True)>
<User(user_id='f1f10d', email='eva.green@ecosoul.net', is_blocked=False)>
<User(user_id='e6376d', email='frank.ocean@seawaves.io', is_blocked=False)>
<User(user_id='b7c921', email='grace.hollow@mistmail.com', is_blocked=False)>
<User(user_id='c3d442', email='henry.vale@ridgeview.net', is_blocked=False)>
<User(user_id='d8e113', email='isla.frost@winterpath.org', is_blocked=True)>
<User(user_id='e2f774', email='jack.mercer@harborline.com', is_blocked=False)>
<User(user_id='f9a305', email='kara.fields@meadowhub.io', is_blocked=False)>
<User(user_id='a1d556', email='liam.cross@ironbridge.co', is_blocked=False)>
<User(user_id='b2e667', email='maya.rivers@bluecurrent.com', is_blocked=True)

In [15]:
with get_session(engine) as session:
    users = session.query(cultpass.User).all()
    for user in users:
        print(user.subscription)

<Subscription(subscription_id='06bc21', user_id='a4ab87', status='active', tier='premium')>
<Subscription(subscription_id='1295c9', user_id='f556c0', status='cancelled', tier='basic')>
<Subscription(subscription_id='106f22', user_id='88382b', status='active', tier='basic')>
<Subscription(subscription_id='4950e6', user_id='888fb2', status='active', tier='basic')>
<Subscription(subscription_id='edf9e5', user_id='f1f10d', status='cancelled', tier='premium')>
<Subscription(subscription_id='6c099b', user_id='e6376d', status='cancelled', tier='basic')>
<Subscription(subscription_id='1b5f0a', user_id='b7c921', status='active', tier='premium')>
<Subscription(subscription_id='38ccca', user_id='c3d442', status='cancelled', tier='basic')>
<Subscription(subscription_id='134397', user_id='d8e113', status='cancelled', tier='basic')>
<Subscription(subscription_id='d707e4', user_id='e2f774', status='cancelled', tier='premium')>
<Subscription(subscription_id='ca44d8', user_id='f9a305', status='active',

In [16]:
with get_session(engine) as session:
    experiences = session.query(cultpass.Experience).all()
    for experience in experiences:
        print(experience)

<Experience(experience_id='39f6bd', title='Carnival History Tour in Olinda', when='2026-05-30 20:10:44.879302')>
<Experience(experience_id='32d3e9', title='Sunset Paddleboarding', when='2026-05-31 20:10:44.890881')>
<Experience(experience_id='7e7605', title='Pelourinho Colonial Walk', when='2026-06-01 20:10:44.890982')>
<Experience(experience_id='f7618d', title='Samba Night at Lapa', when='2026-06-02 20:10:44.891060')>
<Experience(experience_id='8dc778', title='Christ the Redeemer Experience', when='2026-06-03 20:10:44.891131')>
<Experience(experience_id='154184', title='Modern Art at MASP', when='2026-06-04 20:10:44.891211')>
<Experience(experience_id='99ee03', title='Ibirapuera Park Bike Ride', when='2026-06-05 20:10:44.891286')>
<Experience(experience_id='620910', title='Amazon Rainforest Canoe Adventure', when='2026-06-06 20:10:44.891351')>
<Experience(experience_id='746bff', title='Salvador Street Food Tasting', when='2026-06-07 20:10:44.891413')>
<Experience(experience_id='da8bf5